# 🇧🇷 14 Anos de Brasileirão: O que os dados revelam?
**Análise Exploratória de Dados — Série A (2012–2025)**

Neste projeto analisamos **5.320 jogos** do Campeonato Brasileiro Série A ao longo de 14 temporadas completas,
explorando padrões de gols, desempenho dos clubes e o que o mercado de apostas revela sobre o futebol brasileiro.

---

## 0. Imports e Configurações

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

# Cores do projeto
COR_PRINCIPAL = '#1a6b3c'
COR_SECUNDARIA = '#f5c518'
COR_DESTAQUE = '#d62728'

os.makedirs('../images', exist_ok=True)
print('✅ Ambiente configurado!')

## 1. Carregamento e Limpeza dos Dados

In [ ]:
df = pd.read_csv('../data/brasileirao_2012_2025.csv', sep=',', encoding='utf-8')

# Ajustes de tipo
df['data'] = pd.to_datetime(df['data'])
df['temporada'] = df['temporada'].astype(int)
df['mes'] = df['data'].dt.month
df['gols_totais'] = df['gols_casa_ft'] + df['gols_fora_ft']
df['over25'] = (df['gols_totais'] > 2.5).astype(int)
df['btts'] = ((df['gols_casa_ft'] > 0) & (df['gols_fora_ft'] > 0)).astype(int)

# Favorito pela odd média
df['favorito'] = df.apply(
    lambda r: 'Casa' if r['odd_media_casa'] < r['odd_media_fora']
    else ('Fora' if r['odd_media_fora'] < r['odd_media_casa'] else 'Equilibrado'),
    axis=1
)

df['favorito_venceu'] = (
    ((df['favorito'] == 'Casa') & (df['resultado_ft'] == 'H')) |
    ((df['favorito'] == 'Fora') & (df['resultado_ft'] == 'A'))
)

print(f'✅ {len(df):,} jogos carregados | {df["temporada"].nunique()} temporadas | {df["time_casa"].nunique()} clubes únicos')
df.head()

---
## 📦 BLOCO 1 — Panorama Geral

### 1.1 Média de Gols por Temporada — O Brasileirão ficou mais ou menos goleador?

In [ ]:
media_gols = df.groupby('temporada')['gols_totais'].mean().reset_index()
media_gols.columns = ['temporada', 'media_gols']

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(media_gols['temporada'], media_gols['media_gols'], color=COR_PRINCIPAL, edgecolor='white', width=0.6)
ax.axhline(media_gols['media_gols'].mean(), color=COR_DESTAQUE, linestyle='--', linewidth=1.5, label=f'Média geral: {media_gols["media_gols"].mean():.2f}')

for bar, val in zip(bars, media_gols['media_gols']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Média de Gols por Jogo — Brasileirão Série A (2012–2025)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Temporada')
ax.set_ylabel('Média de Gols')
ax.set_ylim(0, 3.5)
ax.legend()
plt.tight_layout()
plt.savefig('../images/01_media_gols_temporada.png', dpi=150)
plt.show()

### 1.2 Distribuição de Resultados — Vitória Casa / Empate / Vitória Fora

In [ ]:
resultado_map = {'H': 'Vitória Casa', 'D': 'Empate', 'A': 'Vitória Fora'}
resultados = df['resultado_ft'].map(resultado_map).value_counts(normalize=True) * 100
resultados = resultados.reindex(['Vitória Casa', 'Empate', 'Vitória Fora'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pizza
cores = [COR_PRINCIPAL, COR_SECUNDARIA, COR_DESTAQUE]
axes[0].pie(resultados, labels=resultados.index, autopct='%1.1f%%', colors=cores,
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Distribuição Geral de Resultados\n(2012–2025)', fontweight='bold')

# Evolução por temporada
evo = df.groupby(['temporada', 'resultado_ft']).size().unstack(fill_value=0)
evo = evo.div(evo.sum(axis=1), axis=0) * 100
evo = evo.rename(columns={'H': 'Vitória Casa', 'D': 'Empate', 'A': 'Vitória Fora'})
evo[['Vitória Casa', 'Empate', 'Vitória Fora']].plot(ax=axes[1], marker='o', linewidth=2,
                                                       color=cores)
axes[1].set_title('Evolução dos Resultados por Temporada (%)', fontweight='bold')
axes[1].set_xlabel('Temporada')
axes[1].set_ylabel('%')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/02_distribuicao_resultados.png', dpi=150)
plt.show()

### 1.3 O Fator Mando de Campo ainda existe?

In [ ]:
mando = df.groupby('temporada').apply(
    lambda x: pd.Series({
        'Vitória Casa (%)': (x['resultado_ft'] == 'H').mean() * 100,
        'Vitória Fora (%)': (x['resultado_ft'] == 'A').mean() * 100,
    })
).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(mando['temporada'], mando['Vitória Casa (%)'], marker='o', color=COR_PRINCIPAL,
        linewidth=2.5, label='Vitória Casa')
ax.plot(mando['temporada'], mando['Vitória Fora (%)'], marker='s', color=COR_DESTAQUE,
        linewidth=2.5, label='Vitória Fora')
ax.fill_between(mando['temporada'], mando['Vitória Casa (%)'], mando['Vitória Fora (%)'],
                alpha=0.1, color=COR_PRINCIPAL)
ax.axvline(2020, color='gray', linestyle=':', linewidth=1.5, label='Pandemia (sem torcida)')

ax.set_title('Fator Mando de Campo — Vitórias em Casa vs Fora (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('% de Jogos')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
plt.tight_layout()
plt.savefig('../images/03_mando_campo.png', dpi=150)
plt.show()

---
## 🏆 BLOCO 2 — Os Grandes Clubes

### 2.1 Ranking Histórico de Ataque e Defesa (2012–2025)

In [ ]:
# Gols marcados e sofridos (home + away)
ataque_casa = df.groupby('time_casa')['gols_casa_ft'].sum()
ataque_fora = df.groupby('time_fora')['gols_fora_ft'].sum()
defesa_casa = df.groupby('time_casa')['gols_fora_ft'].sum()
defesa_fora = df.groupby('time_fora')['gols_casa_ft'].sum()
jogos_casa = df.groupby('time_casa').size()
jogos_fora = df.groupby('time_fora').size()

times = pd.DataFrame({
    'gols_marcados': ataque_casa.add(ataque_fora, fill_value=0),
    'gols_sofridos': defesa_casa.add(defesa_fora, fill_value=0),
    'jogos': jogos_casa.add(jogos_fora, fill_value=0)
}).reset_index().rename(columns={'index': 'time'})

times = times[times['jogos'] >= 100]  # Apenas times com histórico relevante
times['media_gols_marcados'] = times['gols_marcados'] / times['jogos']
times['media_gols_sofridos'] = times['gols_sofridos'] / times['jogos']

top_ataque = times.nlargest(12, 'media_gols_marcados')
top_defesa = times.nsmallest(12, 'media_gols_sofridos')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(top_ataque['time_casa'], top_ataque['media_gols_marcados'],
             color=COR_PRINCIPAL, edgecolor='white')
axes[0].set_title('Top 12 — Melhor Ataque Histórico\n(média de gols marcados por jogo)', fontweight='bold')
axes[0].set_xlabel('Média de Gols Marcados')
axes[0].invert_yaxis()

axes[1].barh(top_defesa['time_casa'], top_defesa['media_gols_sofridos'],
             color=COR_DESTAQUE, edgecolor='white')
axes[1].set_title('Top 12 — Melhor Defesa Histórica\n(média de gols sofridos por jogo)', fontweight='bold')
axes[1].set_xlabel('Média de Gols Sofridos')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../images/04_ranking_ataque_defesa.png', dpi=150)
plt.show()

### 2.2 Quem mais vence fora de casa?

In [ ]:
vitorias_fora = df[df['resultado_ft'] == 'A'].groupby('time_fora').size()
jogos_fora_total = df.groupby('time_fora').size()
pct_vitoria_fora = (vitorias_fora / jogos_fora_total * 100).dropna().reset_index()
pct_vitoria_fora.columns = ['time', 'pct']
pct_vitoria_fora = pct_vitoria_fora[jogos_fora_total >= 50]
top_fora = pct_vitoria_fora.nlargest(12, 'pct')

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_fora['time'], top_fora['pct'], color=COR_SECUNDARIA, edgecolor='white')
for bar, val in zip(bars, top_fora['pct']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_title('Top 12 — Times com Maior % de Vitórias Fora de Casa\n(mínimo 50 jogos fora)', fontweight='bold')
ax.set_xlabel('% de Vitórias como Visitante')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../images/05_vitorias_fora.png', dpi=150)
plt.show()

---
## 📈 BLOCO 3 — Odds vs Realidade

### 3.1 O Favorito Vence com que Frequência?

In [ ]:
df_odds = df.dropna(subset=['odd_media_casa', 'odd_media_fora'])

taxa_favorito = df_odds.groupby('temporada')['favorito_venceu'].mean() * 100

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(taxa_favorito.index, taxa_favorito.values, color=COR_PRINCIPAL, edgecolor='white', width=0.6)
ax.axhline(taxa_favorito.mean(), color=COR_DESTAQUE, linestyle='--', linewidth=1.5,
           label=f'Média geral: {taxa_favorito.mean():.1f}%')
ax.axhline(50, color='gray', linestyle=':', linewidth=1, label='50% (linha base)')

for bar, val in zip(bars, taxa_favorito.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title('O Favorito Vence com que Frequência?\n(baseado na odd média de mercado)', fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('% de Acerto do Favorito')
ax.set_ylim(0, 80)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
plt.tight_layout()
plt.savefig('../images/06_favorito_venceu.png', dpi=150)
plt.show()

### 3.2 Over 2.5 Gols — O Mercado Acerta?

In [ ]:
over_temporada = df.groupby('temporada').agg(
    total_jogos=('over25', 'count'),
    over25_real=('over25', 'mean'),
    btts_real=('btts', 'mean')
).reset_index()

over_temporada['over25_pct'] = over_temporada['over25_real'] * 100
over_temporada['btts_pct'] = over_temporada['btts_real'] * 100

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(over_temporada['temporada'], over_temporada['over25_pct'],
        marker='o', color=COR_PRINCIPAL, linewidth=2.5, label='Over 2.5 Gols')
ax.plot(over_temporada['temporada'], over_temporada['btts_pct'],
        marker='s', color=COR_SECUNDARIA, linewidth=2.5, label='Ambas Marcam (BTTS)')

ax.set_title('Over 2.5 Gols e BTTS por Temporada\nO Brasileirão é um campeonato de muitos gols?',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Temporada')
ax.set_ylabel('% dos Jogos')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
plt.tight_layout()
plt.savefig('../images/07_over25_btts.png', dpi=150)
plt.show()

---
## 🔍 BLOCO 4 — Curiosidades

### 4.1 A Pandemia (2020) Mudou o Comportamento dos Jogos?

In [ ]:
comparativo = df[df['temporada'].isin([2019, 2020, 2021])].groupby('temporada').agg(
    media_gols=('gols_totais', 'mean'),
    pct_vitoria_casa=('resultado_ft', lambda x: (x == 'H').mean() * 100),
    pct_vitoria_fora=('resultado_ft', lambda x: (x == 'A').mean() * 100),
    pct_empate=('resultado_ft', lambda x: (x == 'D').mean() * 100)
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores_anos = [COR_PRINCIPAL, COR_DESTAQUE, '#2196F3']
labels = ['2019\n(com torcida)', '2020\n(sem torcida)', '2021\n(retorno parcial)']

axes[0].bar(labels, comparativo['media_gols'], color=cores_anos, edgecolor='white')
for i, val in enumerate(comparativo['media_gols']):
    axes[0].text(i, val + 0.02, f'{val:.2f}', ha='center', fontweight='bold')
axes[0].set_title('Média de Gols por Jogo\nAntes, Durante e Após a Pandemia', fontweight='bold')
axes[0].set_ylabel('Média de Gols')
axes[0].set_ylim(0, 3)

x = range(len(labels))
axes[1].bar([i - 0.25 for i in x], comparativo['pct_vitoria_casa'], width=0.25,
            label='Vitória Casa', color=COR_PRINCIPAL, edgecolor='white')
axes[1].bar([i for i in x], comparativo['pct_empate'], width=0.25,
            label='Empate', color=COR_SECUNDARIA, edgecolor='white')
axes[1].bar([i + 0.25 for i in x], comparativo['pct_vitoria_fora'], width=0.25,
            label='Vitória Fora', color=COR_DESTAQUE, edgecolor='white')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels)
axes[1].set_title('Distribuição de Resultados\nAntes, Durante e Após a Pandemia', fontweight='bold')
axes[1].set_ylabel('% dos Jogos')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/08_pandemia.png', dpi=150)
plt.show()

### 4.2 Em que mês o Brasileirão tem mais gols?

In [ ]:
meses_nomes = {4: 'Abr', 5: 'Mai', 6: 'Jun', 7: 'Jul', 8: 'Ago',
               9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez', 1: 'Jan', 2: 'Fev', 3: 'Mar'}

gols_mes = df.groupby('mes')['gols_totais'].mean().reset_index()
gols_mes['mes_nome'] = gols_mes['mes'].map(meses_nomes)
gols_mes = gols_mes.sort_values('mes')

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(gols_mes['mes_nome'], gols_mes['gols_totais'],
              color=[COR_DESTAQUE if v == gols_mes['gols_totais'].max() else COR_PRINCIPAL
                     for v in gols_mes['gols_totais']], edgecolor='white')
for bar, val in zip(bars, gols_mes['gols_totais']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Média de Gols por Mês — Há um mês mais goleador?', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Média de Gols por Jogo')
ax.set_ylim(0, 3.2)
plt.tight_layout()
plt.savefig('../images/09_gols_por_mes.png', dpi=150)
plt.show()

---
## 📋 Conclusões

- **Gols:** O Brasileirão mantém uma média estável de aproximadamente X gols por jogo ao longo de 14 temporadas.
- **Mando de campo:** O fator casa ainda existe, mas vem se reduzindo com o tempo.
- **Pandemia:** A ausência de torcida em 2020 impactou visivelmente o mando de campo, aumentando vitórias fora.
- **Favoritos:** O mercado acerta o vencedor em aproximadamente X% dos jogos.
- **Over 2.5:** O Brasileirão não é um campeonato de muitos gols — Over 2.5 ocorre em menos de 50% dos jogos.

---
**Próximo passo:** Integração com API-Football para adicionar estatísticas detalhadas de cada partida (chutes, posse, escanteios, xG).